In [ ]:
# ── CELL 1: Install dependencies ────────────────────────────
!pip install transformers datasets torch -q

In [ ]:
# ── CELL 2: Imports ─────────────────────────────────────────
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# ── CELL 3 (FINAL FIX) ──────────────────────────────────────
import random, json

# Install huggingface_hub to download Xet-stored files properly
# !pip install huggingface_hub -q   ← run this first if not already installed

from huggingface_hub import hf_hub_download

filepath = hf_hub_download(
    repo_id="Hello-SimpleAI/HC3",
    filename="all.jsonl",
    repo_type="dataset"
)

print(f"Downloaded to: {filepath}")

ai_texts, human_texts = [], []

with open(filepath, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        for ans in item.get("chatgpt_answers", []):
            if ans and len(ans.strip()) > 50:
                ai_texts.append(ans.strip())
        for ans in item.get("human_answers", []):
            if ans and len(ans.strip()) > 50:
                human_texts.append(ans.strip())

random.seed(42)
ai_sample    = random.sample(ai_texts,    min(500, len(ai_texts)))
human_sample = random.sample(human_texts, min(500, len(human_texts)))

print(f"Total AI texts found:    {len(ai_texts)}")
print(f"Total Human texts found: {len(human_texts)}")
print(f"\nSampled AI:    {len(ai_sample)}")
print(f"Sampled Human: {len(human_sample)}")
print(f"\nExample AI text:\n{ai_sample[0][:200]}")
print(f"\nExample Human text:\n{human_sample[0][:200]}")

all.jsonl:   0%|          | 0.00/73.7M [00:00<?, ?B/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--Hello-SimpleAI--HC3/snapshots/4d0ff18143b5a7e1b1e79beb540c04549d1e59d3/all.jsonl
Total AI texts found:    26839
Total Human texts found: 57552

Sampled AI:    500
Sampled Human: 500

Example AI text:
Peter J. Denning is a computer scientist and professor emeritus at the United States Naval Academy. He is known for his research and writing in the fields of operating systems, computer networks, and 

Example Human text:
Mathematics in general likes to study the properties of sets by looking for and predicting patterns and symmetries based on underlying structure . * If you tell me how you are made , I will tell you h


In [ ]:
# ── CELL 4: Load RoBERTa detector ───────────────────────────
# This model was fine-tuned specifically to detect ChatGPT text
# Label 0 = Human,  Label 1 = ChatGPT/AI

MODEL_NAME = "Hello-SimpleAI/chatgpt-detector-roberta"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model     = model.to(device)
model.eval()

print(f"Model loaded: {MODEL_NAME}")
print(f"Labels: {model.config.id2label}")

config.json:   0%|          | 0.00/858 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/391 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: Hello-SimpleAI/chatgpt-detector-roberta
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Model loaded: Hello-SimpleAI/chatgpt-detector-roberta
Labels: {0: 'Human', 1: 'ChatGPT'}


In [ ]:
# ── CELL 5: Inference function ───────────────────────────────
def get_predictions(texts, batch_size=16, max_length=512):
    """
    Returns:
        preds      : list of predicted labels (0=human, 1=AI)
        ai_probs   : list of AI-class probabilities
    """
    preds    = []
    ai_probs = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Running inference"):
        batch = texts[i : i + batch_size]

        # Tokenize — truncate long texts to 512 tokens
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            probs   = torch.softmax(outputs.logits, dim=-1)  # [batch, 2]

        # Class 1 = AI/ChatGPT
        ai_prob  = probs[:, 1].cpu().numpy()
        pred     = np.argmax(probs.cpu().numpy(), axis=-1)

        preds.extend(pred.tolist())
        ai_probs.extend(ai_prob.tolist())

    return preds, ai_probs

In [ ]:
# ── CELL 6: Run on AI-generated text ────────────────────────
print("=" * 50)
print("Evaluating AI-generated text (ChatGPT from HC3)...")
print("=" * 50)

ai_preds, ai_confidence = get_predictions(ai_sample)

ai_detected       = sum(p == 1 for p in ai_preds)
ai_detection_rate = ai_detected / len(ai_preds) * 100
avg_ai_confidence = np.mean(ai_confidence) * 100

print(f"\nResults on AI-generated text:")
print(f"  Correctly flagged as AI : {ai_detected} / {len(ai_preds)}")
print(f"  Detection Rate (ASR)    : {ai_detection_rate:.1f}%")
print(f"  Avg AI confidence score : {avg_ai_confidence:.1f}%")

Evaluating AI-generated text (ChatGPT from HC3)...



Running inference: 100%|██████████| 32/32 [00:09<00:00,  3.25it/s]


Results on AI-generated text:
  Correctly flagged as AI : 497 / 500
  Detection Rate (ASR)    : 99.4%
  Avg AI confidence score : 99.3%


In [ ]:
# ── CELL 7: Run on human-written text ───────────────────────
print("\n" + "=" * 50)
print("Evaluating human-written text (HC3 human answers)...")
print("=" * 50)

human_preds, human_confidence = get_predictions(human_sample)

human_wrongly_flagged = sum(p == 1 for p in human_preds)
false_positive_rate   = human_wrongly_flagged / len(human_preds) * 100
avg_human_ai_score    = np.mean(human_confidence) * 100

print(f"\nResults on human-written text:")
print(f"  Wrongly flagged as AI (FP) : {human_wrongly_flagged} / {len(human_preds)}")
print(f"  False Positive Rate        : {false_positive_rate:.1f}%")
print(f"  Avg AI confidence score    : {avg_human_ai_score:.1f}%")


Evaluating human-written text (HC3 human answers)...


Running inference: 100%|██████████| 32/32 [00:10<00:00,  3.02it/s]


Results on human-written text:
  Wrongly flagged as AI (FP) : 4 / 500
  False Positive Rate        : 0.8%
  Avg AI confidence score    : 0.9%


In [ ]:
# ── CELL 8: Summary table ────────────────────────────────────
print("\n" + "=" * 50)
print("BASELINE SUMMARY")
print("=" * 50)
print(f"{'Metric':<40} {'Value':>10}")
print("-" * 52)
print(f"{'Dataset':<40} {'HC3 (all)':>10}")
print(f"{'Detector':<40} {'RoBERTa':>10}")
print(f"{'Samples (AI)':<40} {len(ai_sample):>10}")
print(f"{'Samples (Human)':<40} {len(human_sample):>10}")
print(f"{'AI Detection Rate':<40} {ai_detection_rate:>9.1f}%")
print(f"{'Avg AI confidence (on AI text)':<40} {avg_ai_confidence:>9.1f}%")
print(f"{'False Positive Rate (human text)':<40} {false_positive_rate:>9.1f}%")
print(f"{'Avg AI confidence (on human text)':<40} {avg_human_ai_score:>9.1f}%")


BASELINE SUMMARY
Metric                                        Value
----------------------------------------------------
Dataset                                   HC3 (all)
Detector                                    RoBERTa
Samples (AI)                                    500
Samples (Human)                                 500
AI Detection Rate                             99.4%
Avg AI confidence (on AI text)                99.3%
False Positive Rate (human text)               0.8%
Avg AI confidence (on human text)              0.9%


In [ ]:
# ── CELL 9: Save results to CSV (download from Colab) ────────
results_df = pd.DataFrame({
    "text"       : ai_sample + human_sample,
    "true_label" : ["AI"] * len(ai_sample) + ["Human"] * len(human_sample),
    "pred_label" : ["AI" if p == 1 else "Human" for p in ai_preds + human_preds],
    "ai_prob"    : ai_confidence + human_confidence
})

results_df.to_csv("baseline_results.csv", index=False)
print("\nSaved to baseline_results.csv")

# Quick sanity check on a few examples
print("\nSample predictions:")
print(results_df.head(5).to_string(index=False))


Saved to baseline_results.csv

Sample predictions:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    